# 01 — Explore JSON Dataset Structure

**Milestone 1 | Weather Data Pipeline — DEPI Capstone**

This notebook explores the raw JSON files produced by `src/fetch_data.py`.

By the end of this notebook you will understand:
- The **top-level keys** of each JSON response and what they represent
- The **units** section that defines measurement units for every variable
- The **column-oriented hourly layout** and how to read index-aligned arrays
- How to convert raw JSON into a **pandas DataFrame** (one row per hour)
- How to load and combine data from **all 10 cities**
- The **WMO weather code** reference so `weather_code` values can be interpreted
- The **coordinate snapping** behaviour of the Open-Meteo API

This notebook is the foundation for all M2 transformation work.

---
## 1. Setup & Load Files

We import only the libraries needed for exploration.  
`pathlib.Path` gives OS-independent paths, `json` parses the raw response files, and `pandas` converts arrays to tables.

In [ ]:
import json
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
files = sorted(RAW_DIR.glob("*.json"))

print(f"Found {len(files)} raw JSON files:")
for f in files:
    print(f"  {f.name}")

We expect exactly **10 files** — one per city fetched by `src/fetch_data.py`.  
Each filename follows the pattern `<city>_<UTC-timestamp>.json` so multiple pipeline runs do not overwrite each other.

---
## 2. Top-Level Keys

Once loaded, each JSON file becomes a Python **dictionary**.  
The keys at the root level are the table of contents — let us see what Open-Meteo gives us at the top level.

In [ ]:
# Load the first file as a representative sample
with open(files[0], encoding="utf-8") as f:
    sample = json.load(f)

print(f"File: {files[0].name}")
print(f"\nTop-level keys ({len(sample)}):")
for key, value in sample.items():
    print(f"  {key:<28} → {type(value).__name__}")

### What each top-level key means

| Key | Type | Description |
|-----|------|-------------|
| `latitude` | float | Geographic latitude returned by the API (may differ slightly from requested — see §7) |
| `longitude` | float | Geographic longitude returned by the API |
| `generationtime_ms` | float | Time the API took to generate this response in milliseconds — diagnostic only |
| `utc_offset_seconds` | int | UTC offset in seconds for the requested timezone (always 0 since we pass `timezone=UTC`) |
| `timezone` | str | Timezone string (e.g. `"GMT"`) |
| `timezone_abbreviation` | str | Short form (e.g. `"GMT"`) |
| `elevation` | float | Elevation above sea level in metres at this grid point |
| `hourly_units` | dict | Maps every variable name to its unit of measurement |
| `hourly` | dict | The actual time-series data — one array per variable, all index-aligned |

> The keys we care about for the pipeline are **`latitude`**, **`longitude`**, **`elevation`**, **`hourly_units`**, and **`hourly`**.

In [ ]:
# Print the scalar metadata fields for the sample city
scalar_keys = ["latitude", "longitude", "timezone", "timezone_abbreviation",
               "elevation", "utc_offset_seconds"]

print("Location & metadata:")
for key in scalar_keys:
    if key in sample:
        print(f"  {key:<28}: {sample[key]}")

---
## 3. Units (`hourly_units`)

Before reading any number from `hourly`, we must know its unit.  
`hourly_units` maps every variable name to its unit string — it is the key to interpreting the data correctly.

**Why this matters for M2:** the Postgres schema column names will encode the unit (e.g. `temperature_c`, `wind_speed_kmh`) so the table is self-describing without needing the API docs open.

In [ ]:
hourly_units = sample["hourly_units"]

print("Units reported by Open-Meteo:")
for variable, unit in hourly_units.items():
    print(f"  {variable:<28}: {unit}")

### Unit reference — API name → schema column name

| Variable (API name) | Unit | M2 Schema column name |
|---------------------|------|------------------------|
| `time` | ISO 8601 string | `observed_at` (`TIMESTAMPTZ`) |
| `temperature_2m` | °C | `temperature_c` |
| `relative_humidity_2m` | % | `humidity_pct` |
| `precipitation` | mm | `precipitation_mm` |
| `wind_speed_10m` | km/h | `wind_speed_kmh` |
| `wind_direction_10m` | ° (0–360) | `wind_direction_deg` |
| `pressure_msl` | hPa | `surface_pressure_hpa` |
| `weather_code` | WMO code | `weather_code` (`INT`) |

> `temperature_2m` = temperature at 2 metres above ground. `wind_speed_10m` = at 10 metres above ground. These are the standard WMO measurement heights.

---
## 4. Hourly Data Layout

The `hourly` block is the core of the API response.  
Open-Meteo uses a **column-oriented** (wide) format — each variable is a separate array:

```
hourly = {
    "time":              ["2026-05-03T00:00", "2026-05-03T01:00", ...],  # 168 strings
    "temperature_2m":    [18.5, 18.2, 18.0, ...],                        # 168 floats
    "precipitation":     [0.0, 0.0, 0.1, ...],
    ...
}
```

**Index alignment:** `time[0]`, `temperature_2m[0]`, `precipitation[0]` all refer to the **same hour**.  
168 values = 7 days × 24 hours = one full forecast week per city.

In [ ]:
hourly = sample["hourly"]

print("Hourly variables, array lengths, and first values:")
for key, values in hourly.items():
    print(f"  {key:<28}: {len(values)} values  |  first = {values[0]}")

In [ ]:
# Demonstrate index alignment — all arrays share the same positional index
print("Index-alignment demo (index 0 = first recorded hour):")
idx = 0
print(f"  time[{idx}]                      : {hourly['time'][idx]}")
for key in list(hourly.keys())[1:]:
    unit = hourly_units.get(key, "?")
    print(f"  {key}[{idx}]  : {hourly[key][idx]}  ({unit})")

### Validate: all arrays must have the same length

If any array has a different length the index alignment is broken and the data is corrupted.  
This check should pass for all 10 files — if it fails, re-run `src/fetch_data.py`.

In [ ]:
lengths = {key: len(values) for key, values in hourly.items()}
unique_lengths = set(lengths.values())
all_equal = len(unique_lengths) == 1

print(f"All arrays same length: {all_equal}")
if all_equal:
    n = unique_lengths.pop()
    print(f"  → {n} records = {n // 24} days × 24 hours ✅")
else:
    print("  ⚠️  Array lengths differ — data is misaligned!")
    for k, v in lengths.items():
        print(f"     {k}: {v}")

---
## 5. Single-City DataFrame

Converting the column-oriented `hourly` dict to a pandas DataFrame is the first transform we will automate in M2.  
`pd.DataFrame(hourly)` does the entire conversion in one call — it treats each key as a column and each index position as a row.

In [ ]:
df_single = pd.DataFrame(sample["hourly"])
df_single["time"] = pd.to_datetime(df_single["time"])

print(f"Shape: {df_single.shape[0]} rows × {df_single.shape[1]} columns")
print(f"File : {files[0].name}")
print()
df_single.head(10)

Each row is now **one hour at one city** — exactly the row format needed for the `weather_observations` table in Postgres.

In M2, `normalize.py` will:
1. Call `pd.DataFrame(data["hourly"])` for each city file
2. Rename the columns to their schema names (`temperature_c`, `humidity_pct`, etc.)
3. Add a `location_id` column linking to the `locations` table

---
## 6. Multi-City Combined DataFrame

Now we load all **10 cities** into one combined DataFrame.  
A `city_file` column acts as a temporary identifier before we assign proper `location_id` values in M2.

In [ ]:
all_dfs = []

for filepath in sorted(RAW_DIR.glob("*.json")):
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data["hourly"])
    df["time"]          = pd.to_datetime(df["time"])
    df["city_file"]     = filepath.stem
    df["api_latitude"]  = data["latitude"]
    df["api_longitude"] = data["longitude"]
    df["elevation_m"]   = data["elevation"]
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
print(f"Total rows : {len(combined):,}  ({len(all_dfs)} cities × 168 hours)")
print(f"Columns    : {list(combined.columns)}")
combined.head()

In [ ]:
# Confirm every city contributes exactly 168 rows
city_counts = combined.groupby("city_file")["time"].count()
print("Rows per city:")
print(city_counts.to_string())
print()
assert (city_counts == 168).all(), "⚠️ Some city does not have 168 hours!"
print("✅ All 10 cities have exactly 168 rows.")

---
## 7. Coordinate Snapping

Open-Meteo operates on a fixed resolution grid (~0.25°).  
When we request a coordinate, the API **snaps** it to the nearest grid point and returns the snapped value.  
This means the returned `latitude`/`longitude` will differ slightly from what we requested.

**Design decision for M2:** we will store the **API-returned coordinates** in the `locations` table, not the originally requested ones, because the actual data corresponds to the snapped grid point.

In [ ]:
# Requested coordinates from src/fetch_data.py
REQUESTED = {
    "cairo":      (30.0444,   31.2357),
    "alexandria": (31.2001,   29.9187),
    "london":     (51.5074,   -0.1278),
    "tokyo":      (35.6762,  139.6503),
    "new_york":   (40.7128,  -74.0060),
    "sydney":     (-33.8688, 151.2093),
    "reykjavik":  (64.1466,  -21.9426),
    "mumbai":     (19.0760,   72.8777),
    "sao_paulo":  (-23.5505, -46.6333),
    "cape_town":  (-33.9249,  18.4241),
}

print(f"{'City':<12} {'Req Lat':>9} {'Got Lat':>9} {'ΔLat':>7}  "
      f"{'Req Lon':>10} {'Got Lon':>10} {'ΔLon':>7}")
print("-" * 72)

for city_key, (req_lat, req_lon) in REQUESTED.items():
    rows = combined[combined["city_file"].str.startswith(city_key)]
    if rows.empty:
        print(f"{city_key:<12}  (no file found)")
        continue
    got_lat = rows["api_latitude"].iloc[0]
    got_lon = rows["api_longitude"].iloc[0]
    d_lat = abs(req_lat - got_lat)
    d_lon = abs(req_lon - got_lon)
    print(f"{city_key:<12} {req_lat:>9.4f} {got_lat:>9.4f} {d_lat:>7.4f}  "
          f"{req_lon:>10.4f} {got_lon:>10.4f} {d_lon:>7.4f}")

**Finding:** all deltas are < 0.025°, consistent with Open-Meteo's ~0.25° grid resolution.  
This is expected API behaviour — not a data error. Documented in `docs/data_exploration.md`.

---
## 8. WMO Weather Code Reference

The `weather_code` field uses the **WMO (World Meteorological Organization) code table 4677**.  
Each integer maps to a specific weather condition description.  
We need to know which codes appear in our data and what they mean — this informs the schema design and any future dashboards.

In [ ]:
# WMO code descriptions (full table 4677 — common subset)
WMO_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snowfall",
    73: "Moderate snowfall",
    75: "Heavy snowfall",
    77: "Snow grains",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm (slight or moderate)",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}

# Which codes actually appear in our dataset?
code_counts = (
    combined["weather_code"]
    .value_counts()
    .sort_index()
    .reset_index()
)
code_counts.columns = ["weather_code", "count"]
code_counts["description"] = code_counts["weather_code"].map(
    lambda c: WMO_CODES.get(int(c), "Unknown")
)

print("Weather codes found in dataset:")
print(code_counts.to_string(index=False))

**M2 note:** `weather_code` will be stored as an `INT` column in `weather_observations`.  
A `wmo_codes` lookup table (or application-level dictionary) can decode it for display purposes.

---
## 9. Variable Statistics Overview

A quick descriptive statistics pass across all 10 cities to confirm every variable is in an expected meteorological range.  
This is a preview of the full quality analysis done in `02_data_quality.ipynb`.

In [ ]:
numeric_cols = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "pressure_msl", "weather_code"
]

stats = combined[numeric_cols].describe().T[["min", "max", "mean", "std"]].round(2)
print("Descriptive statistics across all 10 cities (1,680 rows total):")
print(stats.to_string())

### Acceptable range reference (used by M2 `validate.py`)

| Variable | Min | Max | Notes |
|----------|-----|-----|-------|
| `temperature_2m` | −50 °C | +60 °C | Global extremes; our cities should stay well within |
| `relative_humidity_2m` | 0 % | 100 % | Physically impossible outside this range |
| `precipitation` | 0 mm | — | Cannot be negative |
| `wind_speed_10m` | 0 km/h | 200 km/h | Extreme storms approach the upper end |
| `wind_direction_10m` | 0 ° | 360 ° | Compass bearing |
| `pressure_msl` | 870 hPa | 1084 hPa | Recorded global extremes |
| `weather_code` | 0 | 99 | WMO table 4677 values only |

Any row violating these bounds will be flagged or dropped by `src/transform/validate.py` in M2.

---
## 10. Schema Implications Summary

Everything discovered in this notebook drives the M2 schema and transform design.

| Finding | M2 Action |
|---------|----------|
| `hourly` is column-oriented parallel arrays | `normalize.py` uses `pd.DataFrame(data["hourly"])` to produce rows |
| 168 rows per city, consistent array lengths | Assert `len == 168` at ingest; fail loudly if not |
| Timestamps are UTC ISO 8601 strings | Parse with `pd.to_datetime`, store as `TIMESTAMPTZ` in Postgres |
| Coordinates snap to API grid | Store API-returned lat/lon in `locations`, not requested coords |
| `weather_code` is a WMO integer | Store as `INT` column `weather_code` in `weather_observations` |
| Country codes are in `fetch_data.py` config (`"EG"`, `"GB"`, ...) | Store as `country_code CHAR(2)` in `locations` |
| Duplicate rows possible on pipeline retry | `UNIQUE(location_id, observed_at)` + `ON CONFLICT DO UPDATE` |
| Elevation varies per grid point | Store `elevation_m FLOAT` in `locations` from API response |

---
## Summary

**What we learned about the Open-Meteo JSON structure:**

1. **Root keys** — two types: scalar metadata (`latitude`, `longitude`, `elevation`, `timezone`) and two data sections (`hourly_units`, `hourly`).

2. **`hourly_units`** — always check this before reading any value. Maps every variable to its measurement unit (°C, %, mm, km/h, hPa, WMO code).

3. **`hourly` layout** — column-oriented: each variable is a separate array of 168 values. All arrays are index-aligned — position 0 across all variables = the same hour.

4. **DataFrame conversion** — `pd.DataFrame(data["hourly"])` converts the column-oriented format into a row-per-hour table in one call. This is the core of the M2 transform.

5. **10 cities, 1,680 total rows** — every city has exactly 168 hours. Values are within expected meteorological ranges.

6. **Coordinate snapping** — API returns slightly different lat/lon than requested (delta < 0.025°, consistent with ~0.25° grid). Store API-returned coordinates.

7. **`weather_code`** — WMO table 4677 integers. Codes 0–3 (clear/partly cloudy) dominate; 80, 61, 95 (showers, rain, thunderstorm) also appear.

> **This notebook is complete. Proceed to `02_data_quality.ipynb` for null counts, range validation, and full time-coverage checks.**